In [ ]:
%cd "/blue/cai6734/mombongu/ehr project"
!wget -r -N -c -np -nH --cut-dirs=2 -P mimic_demo https://physionet.org/files/mimic-iv-demo-omop/0.9/

In [ ]:
import os, glob, pandas as pd
root = "/blue/cai6734/mombongu/ehr project/mimic_demo"
for f in sorted(glob.glob(root + "/**/*.csv", recursive=True)):
    try:
        df = pd.read_csv(f, nrows=2)
        n = sum(1 for _ in open(f)) - 1
        print(f"\n{os.path.relpath(f, root)}  ({n} rows)")
        print("   ", list(df.columns))
    except Exception as e:
        print("\n", f, "-> could not read:", e)

In [ ]:
import pandas as pd
D = "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv/"
concept = pd.read_csv(D + "2b_concept.csv", usecols=["concept_id", "concept_name"])
names = concept.drop_duplicates("concept_id").set_index("concept_id").concept_name

cond = pd.read_csv(D + "condition_occurrence.csv", usecols=["person_id", "condition_concept_id", "condition_start_date"])
cond["name"] = cond.condition_concept_id.map(names)
print("CONDITIONS rows:", len(cond), "| named:", f"{cond.name.notna().mean():.0%}", "| concept_id 0:", f"{(cond.condition_concept_id == 0).mean():.0%}")
print(cond.groupby("name").person_id.nunique().sort_values(ascending=False).head(25).to_string())

meas = pd.read_csv(D + "measurement.csv", usecols=["person_id", "measurement_concept_id", "value_as_number", "value_source_value"])
meas["name"] = meas.measurement_concept_id.map(names)
print("\nMEASUREMENTS rows:", len(meas), "| named:", f"{meas.name.notna().mean():.0%}")
print(meas.groupby("name").person_id.nunique().sort_values(ascending=False).head(30).to_string())

person = pd.read_csv(D + "person.csv", usecols=["person_id", "gender_source_value", "year_of_birth"])
print("\nPERSON:", person.gender_source_value.value_counts().to_dict(), "| birth years", person.year_of_birth.min(), "to", person.year_of_birth.max())

In [ ]:
import pandas as pd
D = "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv/"
c = pd.read_csv(D + "2b_concept.csv")
print("concept file: ids from", c.concept_id.min(), "to", c.concept_id.max())
print(c.vocabulary_id.value_counts().head(10).to_string())
print(c.domain_id.value_counts().head(10).to_string())
names = c.drop_duplicates("concept_id").set_index("concept_id").concept_name

for fname, std, src, srcval in [
    ("condition_occurrence.csv", "condition_concept_id", "condition_source_concept_id", "condition_source_value"),
    ("measurement.csv", "measurement_concept_id", "measurement_source_concept_id", "measurement_source_value"),
    ("procedure_occurrence.csv", "procedure_concept_id", "procedure_source_concept_id", "procedure_source_value"),
]:
    df = pd.read_csv(D + fname, usecols=[std, src, srcval])
    print(f"\n=== {fname}")
    print("standard id sample:", df[std].head(3).tolist(), "| named by concept file:", f"{df[std].map(names).notna().mean():.0%}")
    print("source id sample:  ", df[src].head(3).tolist(), "| named by concept file:", f"{df[src].map(names).notna().mean():.0%}")
    print("source value sample:", df[srcval].astype(str).head(8).tolist())

In [ ]:
#see lab names
import pandas as pd
D = "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv/"
c = pd.read_csv(D + "2b_concept.csv")
names = c.drop_duplicates("concept_id").set_index("concept_id").concept_name

cond = pd.read_csv(D + "condition_occurrence.csv", usecols=["person_id", "condition_source_concept_id", "condition_source_value", "condition_start_date"])
cond["name"] = cond.condition_source_concept_id.map(names)
print("CONDITIONS named:", f"{cond.name.notna().mean():.0%}")
print(cond.groupby("name").person_id.nunique().sort_values(ascending=False).head(30).to_string())
print("\nTop UNNAMED condition codes:")
print(cond[cond.name.isna()].condition_source_value.str.strip().value_counts().head(10).to_string())

meas = pd.read_csv(D + "measurement.csv", usecols=["person_id", "measurement_source_concept_id", "value_as_number", "value_source_value"])
meas["name"] = meas.measurement_source_concept_id.map(names)
print("\nLABS named:", f"{meas.name.notna().mean():.0%}")
print(meas.groupby("name").person_id.nunique().sort_values(ascending=False).head(40).to_string())
print("\nvalue_source_value sample:", meas.value_source_value.astype(str).head(8).tolist())

In [ ]:
#cell count patients by disease group
import pandas as pd
D = "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv/"
cond = pd.read_csv(D + "condition_occurrence.csv", usecols=["person_id", "condition_source_value"], dtype=str)
code = cond.condition_source_value.str.strip().str.upper()

groups = {
 "Hypertension":        r"^(401|I10)",
 "Hyperlipidemia":      r"^(272|E78)",
 "Diabetes":            r"^(250|E1[0-3])",
 "Atrial fibrillation": r"^(4273|I48)",
 "Coronary disease":    r"^(414|I25)",
 "Heart failure":       r"^(428|I50)",
 "Chronic kidney dis.": r"^(585|N18)",
 "Acute kidney injury": r"^(584|N17)",
 "Anemia":              r"^(285|D5|D62|D63|D64)",
 "Pneumonia":           r"^(48[1-6]|J1[2-8])",
 "Sepsis":              r"^(038|A41|99591|R652)",
 "Hypothyroidism":      r"^(244|E03)",
 "Depression":          r"^(296[23]|F3[23])",
}
rows = {}
for g, rgx in groups.items():
    rows[g] = set(cond[code.str.contains(rgx, regex=True, na=False)].person_id)
for g, s in sorted(rows.items(), key=lambda x: -len(x[1])):
    print(f"{g:22s} {len(s):3d} patients")

per_pt = pd.Series({p: sum(p in s for s in rows.values()) for p in set().union(*rows.values())})
print("\nhow many of these groups does each patient have?")
print(per_pt.value_counts().sort_index().to_string())

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(
    "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv"
)

drug = pd.read_csv(DATA_DIR / "drug_exposure.csv", nrows=5)

print("Columns:")
print(drug.columns.tolist())

print("\nSample rows:")
print(drug.head().to_string())

In [ ]:
DATA_DIR = Path(
    "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9/1_omop_data_csv"
)

drug = pd.read_csv(DATA_DIR / "drug_exposure.csv")

print("=== DATA OVERVIEW ===")
print("Total medication records:", len(drug))
print("Unique patients:", drug["person_id"].nunique())
print("Unique drug concept IDs:", drug["drug_concept_id"].nunique())
print("Missing drug concept IDs:", drug["drug_concept_id"].isna().sum())
print("Missing source concept IDs:", drug["drug_source_concept_id"].isna().sum())

print("\n=== MOST COMMON DRUG CONCEPT IDs ===")
print(
    drug.groupby("drug_concept_id")
    .agg(
        record_count=("drug_exposure_id", "count"),
        patient_count=("person_id", "nunique")
    )
    .sort_values("patient_count", ascending=False)
    .head(15)
)

print("\n=== MOST COMMON SOURCE DRUG VALUES ===")
print(drug["drug_source_value"].value_counts().head(15))

print("\n=== RECORDS PER PATIENT ===")
print(
    drug.groupby("person_id")
    .size()
    .describe()
)

In [ ]:

from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    "/blue/cai6734/mombongu/ehr project/mimic_demo/0.9"
)

# Find available concept/vocabulary files
concept_files = [
    p for p in BASE_DIR.rglob("*.csv")
    if any(word in p.name.lower()
           for word in ["concept", "vocabulary", "relationship"])
]

print("=== CONCEPT / VOCABULARY FILES ===")
for p in concept_files:
    print(p)

# Check the most likely concept table
if concept_files:
    concept_path = next(
        (p for p in concept_files
         if p.name.lower() == "concept.csv"),
        concept_files[0]
    )

    concept = pd.read_csv(concept_path, low_memory=False)

    print("\n=== SELECTED FILE ===")
    print(concept_path)
    print("\nColumns:", concept.columns.tolist())
    print("\nSample rows:")
    print(concept.head(3).to_string(index=False))

    # Try matching the drug concept IDs to concept_id
    if "concept_id" in concept.columns:
        top_ids = drug["drug_concept_id"].value_counts().head(15).index
        matches = concept[
            concept["concept_id"].isin(top_ids)
        ]

        print("\n=== MATCHES FOR COMMON DRUG CONCEPT IDS ===")
        print(matches.to_string(index=False))
        print("\nMatched IDs:", matches["concept_id"].nunique(),
              "out of", len(top_ids))
else:
    print("No concept/vocabulary CSV files found.")


In [ ]:

from pathlib import Path

print("Current folder:", Path.cwd())
print("\nNotebook files:")
for file in Path.cwd().glob("*.ipynb"):
    print(file.name)

In [ ]:
!git clone https://github.com/mombongu/ehr-gen-ai-2026
%cd ehr-gen-ai-2026

In [ ]:

from pathlib import Path

import pandas as pd
import streamlit as st

DATA_DIR = Path(
    "/blue/cai6734/mombongu/ehr project/"
    "mimic_demo/0.9/1_omop_data_csv"
)

st.set_page_config(
    page_title="Clinical Patient Explorer",
    page_icon="🔎",
    layout="wide",
)

st.title("Clinical Patient Explorer")
st.caption(
    "Research prototype | Similar-patient retrieval "
    "and clinical record comparison"
)


@st.cache_data
def load_table(filename):
    path = DATA_DIR / filename
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)


person = load_table("person.csv")
conditions = load_table("condition_occurrence.csv")
procedures = load_table("procedure_occurrence.csv")
measurements = load_table("measurement.csv")
visits = load_table("visit_occurrence.csv")


def patient_ids(df):
    if df.empty or "person_id" not in df.columns:
        return set()
    return set(df["person_id"].dropna())


if person.empty or conditions.empty:
    st.error(
        "Could not load person.csv or condition_occurrence.csv. "
        "Check DATA_DIR and the filenames."
    )
    st.stop()

all_ids = sorted(patient_ids(person), key=str)
aliases = {
    pid: f"Patient {i:03d}"
    for i, pid in enumerate(all_ids, start=1)
}

# Use standardized condition IDs when available.
# Fall back to source values only if a patient has no
# usable standardized condition IDs.
def condition_profile(pid):
    rows = conditions[conditions["person_id"] == pid]

    if "condition_concept_id" in rows.columns:
        ids = pd.to_numeric(
            rows["condition_concept_id"], errors="coerce"
        )
        usable = rows.loc[ids.notna() & (ids != 0)]
        concepts = set(
            pd.to_numeric(
                usable["condition_concept_id"],
                errors="coerce",
            ).dropna().astype(int)
        )
        if concepts:
            return concepts

    if "condition_source_value" in rows.columns:
        values = rows["condition_source_value"].dropna()
        return set(values.astype(str)) - {""}

    return set()


@st.cache_data
def build_profiles(ids):
    return {pid: condition_profile(pid) for pid in ids}


profiles = build_profiles(tuple(all_ids))


def jaccard(a, b):
    union = a | b
    if not union:
        return 0.0
    return len(a & b) / len(union)


def display_values(df, pid, source_col, concept_col):
    if df.empty or "person_id" not in df.columns:
        return pd.DataFrame()

    rows = df[df["person_id"] == pid].copy()
    if rows.empty:
        return pd.DataFrame()

    # Prefer human-readable source values where available.
    if source_col in rows.columns:
        rows["Display"] = rows[source_col].fillna("").astype(str)
    elif concept_col in rows.columns:
        rows["Display"] = rows[concept_col].astype(str)
    else:
        return pd.DataFrame()

    if concept_col in rows.columns and source_col in rows.columns:
        missing = rows["Display"].isin(["", "nan", "None"])
        rows.loc[missing, "Display"] = (
            rows.loc[missing, concept_col].astype(str)
        )

    return rows[rows["Display"].ne("")]


selected_id = st.selectbox(
    "Select a patient",
    all_ids,
    format_func=lambda pid: aliases.get(pid, "Patient"),
)

selected_profile = profiles.get(selected_id, set())

ranked = []
for other_id, other_profile in profiles.items():
    if other_id == selected_id:
        continue

    score = jaccard(selected_profile, other_profile)
    ranked.append({
        "Patient": aliases.get(other_id, "Patient"),
        "ID": other_id,
        "Similarity": score,
        "Shared conditions": len(
            selected_profile & other_profile
        ),
    })

ranked.sort(key=lambda row: row["Similarity"], reverse=True)
top_matches = ranked[:5]

left, middle, right = st.columns(3)
left.metric("Recorded condition concepts", len(selected_profile))
middle.metric("Patients available", len(all_ids))
right.metric("Candidate matches", len(top_matches))

st.divider()
st.subheader("Selected patient's recorded conditions")

selected_conditions = display_values(
    conditions,
    selected_id,
    "condition_source_value",
    "condition_concept_id",
)

if selected_conditions.empty:
    st.info("No displayable condition values were found.")
else:
    st.dataframe(
        selected_conditions[["Display"]].drop_duplicates(),
        use_container_width=True,
        hide_index=True,
    )

st.subheader("Most similar patients")

if top_matches:
    results = pd.DataFrame(top_matches)
    results["Similarity"] = (
        results["Similarity"] * 100
    ).round(1).astype(str) + "%"

    st.dataframe(
        results[["Patient", "Similarity", "Shared conditions"]],
        use_container_width=True,
        hide_index=True,
    )

    match_options = [row["ID"] for row in top_matches]
    match_id = st.selectbox(
        "Inspect a matched patient",
        match_options,
        format_func=lambda pid: aliases.get(pid, "Patient"),
    )

    shared = selected_profile & profiles.get(match_id, set())
    st.write(f"Shared condition concepts: {len(shared)}")

    if shared:
        st.write(sorted(map(str, shared)))

    st.subheader("Recorded procedures for this match")
    procedure_rows = display_values(
        procedures,
        match_id,
        "procedure_source_value",
        "procedure_concept_id",
    )

    if procedure_rows.empty:
        st.info("No displayable procedure values found.")
    else:
        st.dataframe(
            procedure_rows[["Display"]].drop_duplicates(),
            use_container_width=True,
            hide_index=True,
        )

    st.subheader("Recorded measurements for this match")
    measurement_rows = (
        measurements[
            measurements["person_id"] == match_id
        ].copy()
        if not measurements.empty
        and "person_id" in measurements.columns
        else pd.DataFrame()
    )

    if measurement_rows.empty:
        st.info("No measurement records found.")
    else:
        columns = [
            c for c in [
                "measurement_date",
                "measurement_source_value",
                "measurement_concept_id",
                "value_as_number",
                "unit_source_value",
            ]
            if c in measurement_rows.columns
        ]
        st.dataframe(
            measurement_rows[columns].head(100),
            use_container_width=True,
            hide_index=True,
        )

else:
    st.info("No comparable patients found.")

st.caption(
    "Research prototype only. Similarity is based on condition "
    "overlap, not validated clinical equivalence or treatment advice. "
    "Unmapped codes and missing data can affect results."
)


In [ ]:
from pathlib import Path
print(Path("app.py").exists())

In [ ]:
!streamlit run app.py

In [ ]:
import urllib.request

try:
    url = "http://127.0.0.1:8501/_stcore/health"
    response = urllib.request.urlopen(url, timeout=5)
    print("App health:", response.read().decode())
except Exception as e:
    print("Connection test failed:", repr(e))

In [ ]:
import urllib.request

try:
    response = urllib.request.urlopen(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=5
    )
    print(response.status, response.read().decode())
except Exception as e:
    print(type(e).__name__, e)

In [ ]:
import os
from pathlib import Path

import pandas as pd
import streamlit as st

try:
    from openai import OpenAI
except Exception:  # pragma: no cover - optional dependency
    OpenAI = None

st.set_page_config(
    page_title="Clinical Patient Explorer",
    page_icon="🔎",
    layout="wide",
)

DATA_DIR = Path(__file__).parent / "0.9" / "1_omop_data_csv"


@st.cache_data
def load_table(filename):
    path = DATA_DIR / filename
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)


def normalize_patient_ids(series):
    values = pd.to_numeric(series, errors="coerce")
    return sorted(values.dropna().astype(int).unique().tolist())


def get_condition_profile(df, patient_id):
    rows = df[df["person_id"] == patient_id]
    if rows.empty or "condition_concept_id" not in rows.columns:
        return set()
    concept_ids = pd.to_numeric(rows["condition_concept_id"], errors="coerce")
    return set(concept_ids[concept_ids.notna() & (concept_ids != 0)].astype(int).tolist())


def similarity(a, b):
    union = a | b
    return len(a & b) / len(union) if union else 0.0


def summarize_therapy(df, patient_ids):
    if df.empty or df is None:
        return pd.DataFrame(columns=["Therapy", "Similar patients"])

    candidate = df[df["person_id"].isin(patient_ids)].copy()
    if candidate.empty:
        return pd.DataFrame(columns=["Therapy", "Similar patients"])

    label_column = "drug_source_value" if "drug_source_value" in candidate.columns else "drug_concept_id"
    candidate["therapy_label"] = candidate[label_column].fillna("Unknown").astype(str)
    summary = (
        candidate.groupby("therapy_label", as_index=False)["person_id"]
        .nunique()
        .rename(columns={"therapy_label": "Therapy", "person_id": "Similar patients"})
    )
    return summary.sort_values("Similar patients", ascending=False).head(10)


def generate_ai_summary(selected_patient_id, similarity_df, therapy_summary, metric_summary):
    if OpenAI is None:
        return "OpenAI support is not installed. Add the `openai` package and set `OPENAI_API_KEY` to enable this feature."

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        return "Set the `OPENAI_API_KEY` environment variable to enable the optional AI summary."

    top_matches = similarity_df.head(5).copy()
    top_matches["Similarity"] = top_matches["Similarity"].round(3)

    context = {
        "selected_patient_id": selected_patient_id,
        "similarity": top_matches.to_dict(orient="records"),
        "therapy_summary": therapy_summary.head(5).to_dict(orient="records"),
        "outcome_metrics": metric_summary,
    }

    prompt = f"""
    You are a clinical research analyst. Based on the following OMOP-style patient cohort context,
    write a brief, clinician-friendly summary for Patient {selected_patient_id}.
    Keep it concise, avoid making definitive diagnoses, and clearly label this as a research-oriented summary.

    Context:
    {context}
    """

    client = OpenAI(api_key=api_key)

    try:
        if hasattr(client, "responses"):
            response = client.responses.create(
                model="gpt-4o-mini",
                input=[{"role": "user", "content": prompt}],
            )
            if hasattr(response, "output_text") and response.output_text:
                return response.output_text.strip()
            if hasattr(response, "output") and response.output:
                message = response.output[0].content[0].text.value
                return message.strip()
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content.strip()
    except Exception as exc:  # pragma: no cover - depends on external API
        return f"AI summary could not be generated: {exc}"


person = load_table("person.csv")
conditions = load_table("condition_occurrence.csv")
drugs = load_table("drug_exposure.csv")
procedures = load_table("procedure_occurrence.csv")
measurements = load_table("measurement.csv")

st.title("Clinical Patient Explorer")
st.caption("Explore a patient, compare them with a similar cohort, and review their treatment and outcome context.")

if person.empty or conditions.empty:
    st.error(f"Could not find required files in {DATA_DIR}")
    st.stop()

patient_ids = normalize_patient_ids(person["person_id"])
if not patient_ids:
    st.warning("No valid patient IDs were found in `person.csv`.")
    st.stop()

profiles = {pid: get_condition_profile(conditions, pid) for pid in patient_ids}

selected_id = st.selectbox("Patient selector", patient_ids, format_func=lambda pid: f"Patient {pid}")
selected_profile = profiles.get(selected_id, set())

similarity_rows = []
for pid in patient_ids:
    if pid == selected_id:
        continue
    other = profiles.get(pid, set())
    shared = len(selected_profile & other)
    score = similarity(selected_profile, other)
    similarity_rows.append(
        {
            "Patient ID": pid,
            "Similarity": round(score, 3),
            "Shared condition concepts": shared,
        }
    )

similarity_df = pd.DataFrame(similarity_rows)
if not similarity_df.empty:
    similarity_df = similarity_df.sort_values(
        ["Similarity", "Shared condition concepts"],
        ascending=[False, False],
    ).reset_index(drop=True)

left_col, right_col = st.columns([1.1, 1.4])

with left_col:
    st.subheader("Similarity chart")
    if not similarity_df.empty:
        chart_df = similarity_df.head(8).sort_values("Similarity", ascending=True).copy()
        chart_df["Patient ID"] = chart_df["Patient ID"].astype(str)
        st.bar_chart(chart_df.set_index("Patient ID")["Similarity"])
    else:
        st.info("No similar patients were found for this selection.")

with right_col:
    st.subheader("Therapy summary")
    similar_patient_ids = similarity_df["Patient ID"].head(10).tolist() if not similarity_df.empty else []
    therapy_summary = summarize_therapy(drugs, similar_patient_ids)
    if therapy_summary.empty:
        st.info("Drug exposure data for the similar cohort is not available.")
    else:
        st.dataframe(therapy_summary, use_container_width=True, hide_index=True)

    st.subheader("OpenAI summary")
    openai_ready = OpenAI is not None and bool(os.getenv("OPENAI_API_KEY"))
    if not openai_ready:
        st.caption("Optional: install the `openai` package and set `OPENAI_API_KEY` to enable this summary.")

    if st.button("Generate AI summary", type="primary", use_container_width=True, disabled=not openai_ready):
        with st.spinner("Generating a brief cohort summary..."):
            metric_summary = {
                "selected_patient_id": int(selected_id),
                "unique_conditions": len(selected_profile),
                "drug_exposure_records": len(drugs[drugs["person_id"] == selected_id]) if not drugs.empty else 0,
                "procedure_records": len(procedures[procedures["person_id"] == selected_id]) if not procedures.empty else 0,
                "measurement_records": len(measurements[measurements["person_id"] == selected_id]) if not measurements.empty else 0,
                "avg_cohort_similarity": round(similarity_df["Similarity"].mean(), 3) if not similarity_df.empty else 0.0,
            }
            summary = generate_ai_summary(selected_id, similarity_df, therapy_summary, metric_summary)
        st.markdown(summary)

st.subheader("Outcome metrics")
metric_cols = st.columns(4)
metric_cols[0].metric("Unique conditions", len(selected_profile))
metric_cols[1].metric(
    "Drug exposure records",
    len(drugs[drugs["person_id"] == selected_id]) if not drugs.empty else 0,
)
metric_cols[2].metric(
    "Procedure records",
    len(procedures[procedures["person_id"] == selected_id]) if not procedures.empty else 0,
)
metric_cols[3].metric(
    "Measurement records",
    len(measurements[measurements["person_id"] == selected_id]) if not measurements.empty else 0,
)

st.subheader("Similar cohort")
if similarity_df.empty:
    st.info("No comparable patients are available for this patient profile.")
else:
    table_df = similarity_df.head(10).copy()
    table_df["Similarity"] = table_df["Similarity"].round(3)
    st.dataframe(table_df, use_container_width=True, hide_index=True)

st.subheader("Patient detail")
medication_col, procedure_col, measurement_col = st.columns(3)

with medication_col:
    st.caption("Medication records")
    if not drugs.empty:
        rows = drugs[drugs["person_id"] == selected_id].copy()
        cols = [c for c in ["drug_exposure_start_date", "drug_source_value", "drug_concept_id", "quantity"] if c in rows.columns]
        if cols:
            st.dataframe(rows[cols].head(25), use_container_width=True, hide_index=True)
        else:
            st.info("Medication fields are not available for this dataset.")
    else:
        st.info("Drug exposure data not found.")

with procedure_col:
    st.caption("Procedure records")
    if not procedures.empty:
        rows = procedures[procedures["person_id"] == selected_id].copy()
        st.dataframe(rows.head(25), use_container_width=True, hide_index=True)
    else:
        st.info("Procedure data not found.")

with measurement_col:
    st.caption("Measurement records")
    if not measurements.empty:
        rows = measurements[measurements["person_id"] == selected_id].copy()
        cols = [c for c in ["measurement_date", "measurement_source_value", "measurement_concept_id", "value_as_number", "unit_source_value"] if c in rows.columns]
        if cols:
            st.dataframe(rows[cols].head(25), use_container_width=True, hide_index=True)
        else:
            st.info("Measurement fields are not available for this dataset.")
    else:
        st.info("Measurement data not found.")

st.caption("Research prototype. Similarity is based on condition overlap and is not a validated clinical matching method.")

In [ ]:
cd /path/to/ehr-gen-ai-2026
python -m venv .venv
source .venv/bin/activate   # Windows: .venv\Scripts\activate
pip install streamlit pandas openai
streamlit run app.py

In [ ]:
cd ~/path/to/ehr-gen-ai-2026
python3 -m venv .venv
source .venv/bin/activate
pip install streamlit pandas openai
export OPENAI_API_KEY="your_key_here"   # optional, only needed for the AI summary button
streamlit run app.py

In [ ]:
cd "/blue/cai6734/mombongu/ehr project/ehr-gen-ai-2026"

In [ ]:
ls